In [160]:
import math
import pandas as pd
from sqlalchemy import create_engine

In [171]:
df = pd.read_excel("/home/henryx/urban/snp-dwh/xscript_local/visualizador/datos_pnd2425- 12-06-2025_vcf.xlsx",header=2)
df.head(1)


,codigo_,TIPO,EJE,NOMBRE_DEL_OBJETIVO,NOMBRE_DE_LA_POLITICA,META,INDICADOR,FUENTE_DE_INFORMACION,GRUPO_DE_DESAGREGACION,NIVEL_DE_DESAGREGACION,...,LIMITE_INFERIOR,LIMITE_SUPERIOR,COEFICIENTE_DE_VARIACION,NUMERADOR,DENOMINADOR,NOMBRE_DEL_EJE,PERIODICIDAD FICHA METODOLÓGICA,FECHA DE TRANSFERENCIA FICHA METODOLÓGICA,DESAGREGACIÓN FICHA METODOLÓGICA,PERIODICIDAD DEL DATO
0,1.1.1,P,SOCIAL,1. Mejorar las condiciones de vida de la pobla...,1.1 Contribuir a la reducción de la pobreza y ...,Reducir la tasa de pobreza extrema por ingreso...,Tasa de pobreza extrema por ingresos,Instituto Nacional de Estadística y Censos (IN...,Área,Urbana,...,0.040116,0.078344,0.164422,718537.75,12131355.9,Porcentaje,Anual: Estimación puntual de diciembre\nSemest...,Transferencia estimación puntual a junio: hast...,"Nacional, urbano, rural\nSexo",Mensual


In [162]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28714 entries, 0 to 28713
Data columns (total 25 columns):
 #   Column                                     Non-Null Count  Dtype         
---  ------                                     --------------  -----         
 0   codigo_                                    28714 non-null  object        
 1   TIPO                                       28714 non-null  object        
 2   EJE                                        28714 non-null  object        
 3   NOMBRE_DEL_OBJETIVO                        28714 non-null  object        
 4   NOMBRE_DE_LA_POLITICA                      28714 non-null  object        
 5   META                                       28714 non-null  object        
 6   INDICADOR                                  28713 non-null  object        
 7   FUENTE_DE_INFORMACION                      28713 non-null  object        
 8   GRUPO_DE_DESAGREGACION                     28714 non-null  object        
 9   NIVEL_DE_DESAGREG

## LIMPIEZA

### EJE CON MAYUSCULAS PARA EVITAR DUPLICADOS

In [163]:
# Convertir EJE a mayúsculas
if 'EJE' in df.columns:
    df['EJE'] = df['EJE'].str.upper()

In [164]:
df['EJE'].value_counts()

EJE
SOCIAL                                       18077
INFRAESTRUCTURA, ENERGÍA Y MEDIO AMBIENTE     4973
INSTITUCIONAL                                 2278
DESARROLLO ECONÓMICO                          2190
GESTIÓN DE RIESGOS                            1196
Name: count, dtype: int64

In [165]:
import re

# Función para remover el prefijo numérico (como "1. texto...")
def clean_text_prefix(text):
    if not isinstance(text, str):
        return text
    return re.sub(r'^\s*\d+\.\s*', '', text).strip()

# Limpiar prefijos en columnas específicas
cols_to_clean = ['NOMBRE_DEL_OBJETIVO', 'NOMBRE_DE_LA_POLITICA']
for col in cols_to_clean:
    if col in df.columns:
        df[col] = df[col].apply(clean_text_prefix)

In [166]:
df["NOMBRE_DE_LA_POLITICA"].value_counts()

NOMBRE_DE_LA_POLITICA
1 Prever, prevenir y controlar, con pertinencia territorial, los fenómenos de violencia y delincuencia que afectan a la ciudadanía y sus derechos, fortaleciendo la convivencia pacífica.                                                                                                                                                                           8318
2 Optimizar las infraestructuras construidas, capacidades instaladas y de gestión del transporte multimodal, para una movilización nacional e internacional de personas, bienes y mercancías de manera sostenible, oportuna y segura.                                                                                                                               3023
1 Garantizar el acceso universal a una educación, inclusiva, equitativa, pertinente e intercultural para niños, niñas, adolescentes, jóvenes y adultos, promoviendo la permanencia y culminación de sus estudios; y asegurando su movilidad dentro del Sistema N

In [167]:
# Función para limpiar prefijos numéricos como "1.", "1 ", " 2. ", etc.
def clean_numeric_prefix(text):
    if not isinstance(text, str):
        return text
    return re.sub(r'^\s*\d+[\.\s]*', '', text).strip()

# Limpiar prefijos numéricos en columnas clave
cols_to_clean = ['NOMBRE_DEL_OBJETIVO', 'NOMBRE_DE_LA_POLITICA', 'META', 'INDICADOR']
for col in cols_to_clean:
    if col in df.columns:
        df[col] = df[col].apply(clean_numeric_prefix)

In [168]:
# Función para limpiar valores NaN -> None
def clean_val(v):
    if pd.isna(v):  # maneja None, np.nan, pd.NA
        return None
    return v

# Función para limpiar valores numéricos, convierte strings problemáticos a None
def clean_numeric_val(v):
    if pd.isna(v):
        return None
    if isinstance(v, str):
        v_strip = v.strip()
        if v_strip in ('', '-', 'NaN', 'nan', '(omitted)'):
            return None
        try:
            return float(v_strip)
        except ValueError:
            return None
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

In [169]:
safasdfsadf

NameError: name 'safasdfsadf' is not defined

In [ ]:
df["ESTIMADOR"] = pd.to_numeric(df["ESTIMADOR"], errors="coerce")
df["ERROR_ESTANDAR"] = pd.to_numeric(df["ERROR_ESTANDAR"], errors="coerce")   
df["LIMITE_INFERIOR"] = pd.to_numeric(df["LIMITE_INFERIOR"], errors="coerce")
df["LIMITE_SUPERIOR"] = pd.to_numeric(df["LIMITE_SUPERIOR"], errors="coerce")
df["COEFICIENTE_DE_VARIACION"] = pd.to_numeric(df["COEFICIENTE_DE_VARIACION"], errors="coerce")
df["DENOMINADOR"] = pd.to_numeric(df["DENOMINADOR"], errors="coerce")

In [ ]:
df.shape

(28714, 25)

In [ ]:
cols = ['NOMBRE_DEL_OBJETIVO', 'NOMBRE_DE_LA_POLITICA', 'META', 'FUENTE_DE_INFORMACION']
for col in cols:
    df[col] = df[col].astype('string').apply(lambda x: x if pd.isna(x) or str(x).strip().endswith('.') else str(x).strip() + '.')

In [ ]:
df['FUENTE_DE_INFORMACION'].value_counts(dropna=False).sum()

np.int64(28714)

In [ ]:
df['codigo_'] = df['codigo_'].astype('string')
df['TIPO'] = df['TIPO'].astype('string')
df['EJE'] = df['EJE'].astype('string')
df['NOMBRE_DEL_OBJETIVO'] = df['NOMBRE_DEL_OBJETIVO'].astype('string')
df['NOMBRE_DE_LA_POLITICA'] = df['NOMBRE_DE_LA_POLITICA'].astype('string')
df['META'] = df['META'].astype('string')
df['INDICADOR'] = df['INDICADOR'].astype('string')
df['FUENTE_DE_INFORMACION'] = df['FUENTE_DE_INFORMACION'].astype('string')
df['GRUPO_DE_DESAGREGACION'] = df['GRUPO_DE_DESAGREGACION'].astype('string')
df['NIVEL_DE_DESAGREGACION'] = df['NIVEL_DE_DESAGREGACION'].astype('string')
df['CODIGO_GEOGRAFICO_DPA'] = df['CODIGO_GEOGRAFICO_DPA'].astype('string')
df['MES_ANIO'] = df['MES_ANIO'].astype('string')
df['NOMBRE_DEL_EJE'] = df['NOMBRE_DEL_EJE'].astype('string')
df['PERIODICIDAD FICHA METODOLÓGICA'] = df['PERIODICIDAD FICHA METODOLÓGICA'].astype('string')
df['FECHA DE TRANSFERENCIA FICHA METODOLÓGICA'] = df['FECHA DE TRANSFERENCIA FICHA METODOLÓGICA'].astype('string')
df['DESAGREGACIÓN FICHA METODOLÓGICA '] = df['DESAGREGACIÓN FICHA METODOLÓGICA '].astype('string')
df['PERIODICIDAD DEL DATO'] = df['PERIODICIDAD DEL DATO'].astype('string')

In [ ]:

# Parámetros de conexión a PostgreSQL
usuario = "postgres"
password = "123456"
host = "localhost"
puerto = "5432"
base_datos = "SNP"

# Crear engine de conexión
engine = create_engine(f"postgresql+psycopg2://{usuario}:{password}@{host}:{puerto}/{base_datos}")

In [ ]:
# Parámetros de conexión a PostgreSQL
usuario = "postgres"
password = "JHEQWR2ZUASDasdgASd98x"
host = "155.138.253.162"
puerto = "5432"
base_datos = "test_snp_pnd"

## SQL TABLAS

In [ ]:
import pandas as pd
from sqlalchemy import text
from sqlalchemy.exc import SQLAlchemyError

def get_or_insert_id(engine, table, where_cols, values, id_field='id'):
    where_clauses = []
    params = {}
    for col in where_cols:
        val = values.get(col)
        if val is None:
            where_clauses.append(f"{col} IS NULL")
        else:
            where_clauses.append(f"{col} = :{col}")
            params[col] = val
    where_clause = ' AND '.join(where_clauses)

    insert_cols = ', '.join(values.keys())
    insert_vals = ', '.join([f":{k}" for k in values])

    with engine.begin() as conn:
        # Buscar si existe
        result = conn.execute(
            text(f"SELECT {id_field} FROM {table} WHERE {where_clause}"),
            params
        ).fetchone()
        if result:
            return result[0]

        # Insertar si no existe
        try:
            result = conn.execute(
                text(f"INSERT INTO {table} ({insert_cols}) VALUES ({insert_vals}) RETURNING {id_field}"),
                values
            ).fetchone()
            return result[0]
        except SQLAlchemyError as e:
            print(f"[ERROR] Al insertar en {table}: {e}")
            return None

# -----------------------------
# Limpieza de strings en todo el DataFrame
cols_str = df.columns
df[cols_str] = df[cols_str].apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# Diccionarios para claves
mapas = {
    'eje': {}, 'objetivo': {}, 'politica': {}, 'meta': {},
    'indicador': {}, 'fuente': {}, 'desagregacion': {}, 'tiempo': {}
}



In [ ]:
co = 0
datas =[]
for _, row in df[['GRUPO_DE_DESAGREGACION', 'NIVEL_DE_DESAGREGACION', 'CODIGO_GEOGRAFICO_DPA']].dropna(subset=['GRUPO_DE_DESAGREGACION', 'NIVEL_DE_DESAGREGACION']).drop_duplicates().iterrows():
    codigo_geo = str(row['CODIGO_GEOGRAFICO_DPA']) if pd.notna(row['CODIGO_GEOGRAFICO_DPA']) else None
    datas.append(row)
    key = (row['GRUPO_DE_DESAGREGACION'], row['NIVEL_DE_DESAGREGACION'], codigo_geo)
    valores = {'grupo_desagregacion': key[0], 'nivel_desagregacion': key[1], 'codigo_geografico_dpa': key[2]}
    co = co+1

In [ ]:
co = 0
for _, row in df[['GRUPO_DE_DESAGREGACION', 'NIVEL_DE_DESAGREGACION', 'CODIGO_GEOGRAFICO_DPA']].dropna(subset=['GRUPO_DE_DESAGREGACION', 'NIVEL_DE_DESAGREGACION']).drop_duplicates().iterrows():
    codigo_geo = str(row['CODIGO_GEOGRAFICO_DPA']) if pd.notna(row['CODIGO_GEOGRAFICO_DPA']) else None
    key = (row['GRUPO_DE_DESAGREGACION'], row['NIVEL_DE_DESAGREGACION'], codigo_geo)
    valores = {'grupo_desagregacion': key[0], 'nivel_desagregacion': key[1], 'codigo_geografico_dpa': key[2]}
    #mapas['desagregacion'][key] = get_or_insert_id(engine, 'desagregacion', list(valores.keys()), valores, 'id_desagregacion')
    co=co+1

In [ ]:
for _, row in df[['codigo_', 'INDICADOR']].dropna().drop_duplicates().iterrows():
    id_meta = row['codigo_']
    if mapas['meta'].get(id_meta):
        id_ind = f"ind_{id_meta}"
        valores = {'id_indicador': id_ind, 'nombre_indicador': row['INDICADOR'], 'id_meta': id_meta}
        #mapas['indicador'][id_ind] = 1#get_or_insert_id(engine, 'indicador', ['id_indicador'], valores, 'id_indicador')


In [178]:
for _, row in df[['codigo_', 'INDICADOR']].dropna().drop_duplicates().iterrows():
    id_meta = row['codigo_']
    if mapas['meta'].get(id_meta):
        id_ind = f"ind_{id_meta}"
        valores = {'id_indicador': id_ind, 'nombre_indicador': row['INDICADOR'], 'id_meta': id_meta}
        print(id_ind)


In [180]:
df[['codigo_', 'INDICADOR']].dropna().drop_duplicates()

,codigo_,INDICADOR
0,1.1.1,Tasa de pobreza extrema por ingresos
12,1.2.1,Tasa de pobreza por necesidades básicas insati...
16,1.7.1,Prevalencia de desnutrición crónica en niñas y...
61,1.3.1,Cobertura de vacunación de Rotavirus
69,1.3.2,"Cobertura de vacunación de SRP (Sarampión, Rub..."
...,...,...
25717,9.7.1,Montos de Cooperación Internacional No Reembol...
25729,9.8.1,Posicionamiento del Ecuador en el ranking de p...
25742,9.9.1,Índice de capacidad operativa promedio de los ...
27518,10.1.1,Índice de Fortalecimiento de la gobernanza loc...


In [175]:
valores

{'grupo_desagregacion': 'Cantonal',
 'nivel_desagregacion': 'Mejía ',
 'codigo_geografico_dpa': '1703'}

In [ ]:
df[['GRUPO_DE_DESAGREGACION', 'NIVEL_DE_DESAGREGACION', 'CODIGO_GEOGRAFICO_DPA']].dropna(subset=['GRUPO_DE_DESAGREGACION', 'NIVEL_DE_DESAGREGACION']).drop_duplicates()['GRUPO_DE_DESAGREGACION'].value_counts()

GRUPO_DE_DESAGREGACION
Cantonal                                      294
Municipal                                     188
Otros ámbitos                                  81
Provincial                                     41
Base indexada                                  39
Deporte                                        31
Provincia sede matriz de las instituciones     27
Grupos de edad                                 26
Provincia                                      26
Provincia de residencia del estudiante         25
Geográfico                                     25
Provicial                                      25
Grupos etarios                                 24
Provincial                                     24
Área de conocimiento                           22
Grupos de Edad                                 22
Etnia                                          19
Ámbito de fomento                              10
Zonas de planificación                          9
Demarcación Hidrográfica   

In [ ]:
# EJE
for nombre in df['EJE'].dropna().unique():
    id_eje = get_or_insert_id(engine, 'eje', ['nombre_eje'], {'nombre_eje': nombre})
    mapas['eje'][nombre] = id_eje

# OBJETIVO
df_obj = df[['codigo_', 'NOMBRE_DEL_OBJETIVO', 'EJE']].dropna().drop_duplicates()
for _, row in df_obj.iterrows():
    id_obj = row['codigo_'].split('.')[0] + '.'
    id_eje = mapas['eje'].get(row['EJE'])
    if id_eje:
        valores = {'id_objetivo': id_obj, 'nombre_objetivo': row['NOMBRE_DEL_OBJETIVO'], 'id_eje': id_eje}
        mapas['objetivo'][id_obj] = get_or_insert_id(engine, 'objetivo', ['id_objetivo'], valores, 'id_objetivo')

# POLITICA
df_pol = df[['codigo_', 'NOMBRE_DE_LA_POLITICA']].dropna().drop_duplicates()
for _, row in df_pol.iterrows():
    id_pol = '.'.join(row['codigo_'].split('.')[:2])
    id_obj = mapas['objetivo'].get(row['codigo_'].split('.')[0] + '.')
    if id_obj:
        valores = {'id_politica': id_pol, 'nombre_politica': row['NOMBRE_DE_LA_POLITICA'], 'id_objetivo': id_obj}
        mapas['politica'][id_pol] = get_or_insert_id(engine, 'politica', ['id_politica'], valores, 'id_politica')

# META
for _, row in df[['codigo_', 'META']].dropna().drop_duplicates().iterrows():
    id_pol = '.'.join(row['codigo_'].split('.')[:2])
    if mapas['politica'].get(id_pol):
        valores = {'id_meta': row['codigo_'], 'descripcion_meta': row['META'], 'id_politica': id_pol}
        mapas['meta'][row['codigo_']] = get_or_insert_id(engine, 'meta', ['id_meta'], valores, 'id_meta')

# INDICADOR
for _, row in df[['codigo_', 'INDICADOR']].dropna().drop_duplicates().iterrows():
    id_meta = row['codigo_']
    if mapas['meta'].get(id_meta):
        id_ind = f"ind_{id_meta}"
        valores = {'id_indicador': id_ind, 'nombre_indicador': row['INDICADOR'], 'id_meta': id_meta}
        mapas['indicador'][id_ind] = get_or_insert_id(engine, 'indicador', ['id_indicador'], valores, 'id_indicador')

# FUENTE DE INFORMACION
for fuente in df['FUENTE_DE_INFORMACION'].dropna().unique():
    mapas['fuente'][fuente] = get_or_insert_id(engine, 'fuente_informacion', ['nombre_fuente'], {'nombre_fuente': fuente})

# DESAGREGACION
for _, row in df[['GRUPO_DE_DESAGREGACION', 'NIVEL_DE_DESAGREGACION', 'CODIGO_GEOGRAFICO_DPA']].dropna(subset=['GRUPO_DE_DESAGREGACION', 'NIVEL_DE_DESAGREGACION']).drop_duplicates().iterrows():
    codigo_geo = str(row['CODIGO_GEOGRAFICO_DPA']) if pd.notna(row['CODIGO_GEOGRAFICO_DPA']) else None
    key = (row['GRUPO_DE_DESAGREGACION'], row['NIVEL_DE_DESAGREGACION'], codigo_geo)
    valores = {'grupo_desagregacion': key[0], 'nivel_desagregacion': key[1], 'codigo_geografico_dpa': key[2]}
    mapas['desagregacion'][key] = get_or_insert_id(engine, 'desagregacion', list(valores.keys()), valores, 'id_desagregacion')

# TIEMPO
df_tiempo = df[['MES_ANIO', 'FECHA', 'PERIODICIDAD FICHA METODOLÓGICA', 'FECHA DE TRANSFERENCIA FICHA METODOLÓGICA', 'PERIODICIDAD DEL DATO']].dropna(subset=['MES_ANIO']).drop_duplicates()
for _, row in df_tiempo.iterrows():
    fecha_val = pd.to_datetime(row['FECHA']) if pd.notna(row['FECHA']) else None
    transferencia_val = str(row['FECHA DE TRANSFERENCIA FICHA METODOLÓGICA']) if pd.notna(row['FECHA DE TRANSFERENCIA FICHA METODOLÓGICA']) else None

    periodicidad_ficha = clean_val(row['PERIODICIDAD FICHA METODOLÓGICA'])
    periodicidad_dato = clean_val(row['PERIODICIDAD DEL DATO'])

    key = (row['MES_ANIO'], fecha_val, periodicidad_ficha, transferencia_val, periodicidad_dato)
    valores = {
        'mes_anio': key[0],
        'fecha': key[1],
        'periodicidad_ficha_metodologica': key[2],
        'fecha_transferencia_ficha_metodologica': key[3],
        'periodicidad_dato': key[4]
    }
    mapas['tiempo'][key] = get_or_insert_id(engine, 'tiempo', list(valores.keys()), valores, 'id_tiempo')

# MEDICIONES
for _, row in df.dropna(subset=['INDICADOR', 'MES_ANIO']).iterrows():
    try:
        id_ind = f"ind_{row['codigo_']}"
        codigo_geo = str(row['CODIGO_GEOGRAFICO_DPA']) if pd.notna(row['CODIGO_GEOGRAFICO_DPA']) else None
        id_desag = mapas['desagregacion'].get((row['GRUPO_DE_DESAGREGACION'], row['NIVEL_DE_DESAGREGACION'], codigo_geo))

        fecha_val = pd.to_datetime(row['FECHA']) if pd.notna(row['FECHA']) else None
        transferencia_val = str(row['FECHA DE TRANSFERENCIA FICHA METODOLÓGICA']) if pd.notna(row['FECHA DE TRANSFERENCIA FICHA METODOLÓGICA']) else None

        periodicidad_ficha = clean_val(row['PERIODICIDAD FICHA METODOLÓGICA'])
        periodicidad_dato = clean_val(row['PERIODICIDAD DEL DATO'])

        tiempo_key = (row['MES_ANIO'], fecha_val, periodicidad_ficha, transferencia_val, periodicidad_dato)
        id_tiempo = mapas['tiempo'].get(tiempo_key)
        id_fuente = mapas['fuente'].get(row['FUENTE_DE_INFORMACION'])

        if id_ind and id_desag and id_tiempo:
            data = {
                'id_indicador': id_ind,
                'id_desagregacion': id_desag,
                'id_tiempo': id_tiempo,
                'id_fuente': id_fuente,
                'estimador': clean_numeric_val(row.get('ESTIMADOR')),
                'error_estandar': clean_numeric_val(row.get('ERROR_ESTANDAR')),
                'limite_inferior': clean_numeric_val(row.get('LIMITE_INFERIOR')),
                'limite_superior': clean_numeric_val(row.get('LIMITE_SUPERIOR')),
                'coeficiente_variacion': clean_numeric_val(row.get('COEFICIENTE_DE_VARIACION')),
                'numerador': clean_numeric_val(row.get('NUMERADOR')),
                'denominador': clean_numeric_val(row.get('DENOMINADOR')),
                'nombre_del_eje': clean_val(row.get('NOMBRE_DEL_EJE')),
                'tipo': row.get('TIPO')            
            }
            get_or_insert_id(engine, 'mediciones', ['id_indicador', 'id_desagregacion', 'id_tiempo'], data)
        else:
            print(f"[WARN] Faltan claves en fila {row['codigo_']}")
    except Exception as e:
        print(f"[ERROR] Fila {row.get('codigo_', 'SIN_CODIGO')}: {e}")

print("\u2705 Carga incremental completada exitosamente.")